# Med-Nexus — Vast.ai Reproducible Training Notebook

**Purpose:** run the real medical-data experiments after connecting a Vast.ai GPU.

This notebook is intentionally reproducible. It does **not** manufacture benchmark results. Every final metric must come from an executed training/evaluation run.


In [ ]:
# GPU verification
import os, subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


In [ ]:
# Install project dependencies
!python -m pip install -r requirements.txt


In [ ]:
# Verify repository
!python -m pytest tests/ -v


## Dataset preparation

Prepare a patient-level manifest with columns:

`image_path,label,patient_id`

Do not put patient-identifying information into the repository. Keep the raw dataset outside Git.


In [ ]:
from pathlib import Path
import pandas as pd
manifest = Path("data/full/manifest.csv")
if manifest.exists():
    df = pd.read_csv(manifest)
    required = {"image_path", "label", "patient_id"}
    missing = required - set(df.columns)
    print(f"Rows: {len(df):,}")
    print("Missing columns:", missing)
    print("Unique patients:", df.patient_id.nunique())
else:
    print("Create data/full/manifest.csv before training.")


## Sanity training

Run one epoch before spending GPU budget on the full study.


In [ ]:
!python train.py --manifest data/full/manifest.csv --root-dir data/full --epochs 1 --batch-size 8 --lr 1e-4 --out checkpoints/sanity.pt


## Full training

Run each model size using a separate configuration. Record the exact measured parameter count.


In [ ]:
# Example large-model run. Adjust batch size to the available GPU.
!python train.py --manifest data/full/manifest.csv --root-dir data/full --epochs 10 --batch-size 8 --lr 1e-4 --out checkpoints/med_nexus_large.pt


## Held-out evaluation

The test manifest must remain untouched during model selection.


In [ ]:
!python evaluate.py --manifest data/test/manifest.csv --root-dir data/test --checkpoint checkpoints/med_nexus_large.pt --out results/test_metrics.json


## Smoke/engineering experiments

The following command validates the experiment orchestration. For publication results, the experiment modules must be connected to the locked real checkpoint/test manifest and their outputs must be saved separately from smoke results.


In [ ]:
!python experiments/run_all_experiments.py --smoke


## Experiment bookkeeping

For every real run save:

- experiment ID
- Git commit
- dataset identifier
- split identifier
- seed
- parameter count
- GPU model/VRAM
- CUDA/PyTorch versions
- hyperparameters
- training time
- checkpoint hash
- raw metrics


In [ ]:
import subprocess, json, platform, torch, pathlib, hashlib
info = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "git_commit": subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
}
print(json.dumps(info, indent=2))


## Packaging

After the real experiments, package code, configs, metrics, figures and documentation. Do not package restricted raw medical data or credentials.


In [ ]:
!git status --short
